# End-to-end audio-to-graph inference demo

This notebook performs the faculty-required audio sample -> graph inference -> text output path. In the full project it loads one real held-out GTZAN WAV and constructs its graph live. The faculty ZIP omits licensed raw audio, so the same notebook has a clearly labelled fallback to the included graph for that exact track.

In [1]:
from pathlib import Path
import json, sys
import torch
import yaml

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.audio_features import compute_rich_segment_features, load_and_resample
from src.gnn_model import GNNClassifier
from src.graph_builder import build_segment_graph

config = yaml.safe_load((ROOT / 'config.yaml').read_text(encoding='utf-8'))
checkpoint_path = ROOT / 'checkpoints/task2_gnn_best.pt'
splits = json.loads((ROOT / 'data/splits/gtzan_splits.json').read_text(encoding='utf-8'))
track_id = splits['test'][0]
true_genre = track_id.split('.')[0]
audio_path = ROOT / 'data/raw/gtzan/Data/genres_original' / true_genre / f'{track_id}.wav'
demo_graph_path = ROOT / 'demo_assets/gtzan_demo_graph.pt'
if not demo_graph_path.exists():
    demo_graph_path = ROOT / 'data/processed/gtzan_graphs' / f'{track_id}.pt'
assert checkpoint_path.exists(), 'The included Task 2 checkpoint is missing.'
assert audio_path.exists() or demo_graph_path.exists(), 'Neither real audio nor its graph fallback is available.'

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
if audio_path.exists():
    sample_rate = int(config['audio']['sample_rate'])
    audio = load_and_resample(str(audio_path), sr=sample_rate)
    features = compute_rich_segment_features(
        audio, sr=sample_rate,
        window_sec=float(config['segmentation']['gtzan_window_sec']),
        hop_length=int(config['audio']['mel_hop_length']),
        n_fft=int(config['audio']['mel_n_fft']), normalize=True,
    )
    graph_data = build_segment_graph(
        features, tau=float(config['graph']['tau']),
        temporal_edges=bool(config['graph']['temporal_edges']),
        self_loops=bool(config['graph']['self_loops']),
        directed=bool(config['graph']['directed']),
    )
    graph = {
        'x': graph_data.x, 'edge_index': graph_data.edge_index,
        'y': torch.tensor(checkpoint['genres'].index(true_genre)),
        'metadata': {'track_id': track_id, 'genre': true_genre},
    }
    input_source = f'live raw WAV ({len(audio) / sample_rate:.2f} s at {sample_rate} Hz)'
else:
    graph = torch.load(demo_graph_path, map_location='cpu', weights_only=False)
    input_source = 'included real preprocessed graph fallback (licensed WAV omitted from ZIP)'

model = GNNClassifier(
    input_dim=int(checkpoint['feature_dim']),
    hidden_dim=int(config['task2']['graph_model']['hidden_dim']),
    num_layers=int(config['task2']['graph_model']['num_layers']),
    num_classes=len(checkpoint['genres']),
    dropout=float(config['task2']['graph_model']['dropout']),
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

GNNClassifier(
  (encoder): GraphSAGEEncoder(
    (convs): ModuleList(
      (0): SAGEConv(77, 256, aggr=mean)
      (1-2): 2 x SAGEConv(256, 256, aggr=mean)
    )
    (bns): ModuleList(
      (0-2): 3 x BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
  )
  (classifier): Sequential(
    (0): Dropout(p=0.3, inplace=False)
    (1): Linear(in_features=256, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.15, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [3]:
node_batch = torch.zeros(graph['x'].shape[0], dtype=torch.long, device=device)
with torch.no_grad():
    logits = model(graph['x'].to(device), graph['edge_index'].to(device), node_batch)
probabilities = torch.softmax(logits, dim=1)[0].cpu()
top_indices = probabilities.topk(3).indices.tolist()
genres = checkpoint['genres']
true_index = int(graph['y'])
print('Input source:', input_source)
print('Graph:', graph['x'].shape[0], 'nodes /', graph['edge_index'].shape[1], 'edges /', graph['x'].shape[1], 'features')
print('Track:', graph['metadata']['track_id'])
print('Ground-truth genre:', genres[true_index])
print('Top predictions:', [(genres[i], float(probabilities[i])) for i in top_indices])

Input source: live raw WAV (30.01 s at 22050 Hz)
Graph: 6 nodes / 16 edges / 77 features
Track: jazz.00015
Ground-truth genre: jazz
Top predictions: [('blues', 0.4442630708217621), ('disco', 0.3172651529312134), ('country', 0.09909915924072266)]


The saved output above is produced from the real local WAV through resampling, 77-dimensional segment feature extraction, temporal/similarity graph construction, GraphSAGE inference, and text output. When executed from the faculty ZIP, the notebook uses the included graph for the same real held-out track because licensed raw GTZAN audio is intentionally not redistributed.